# Introduction to Embeddings and Tokenization

In this notebook, we will explore definitions of the fundamental building blocks of Large Language Models (LLMs):
1.  **Tokenization**: How text is broken down into numbers.
2.  **Embeddings**: How those numbers are converted into meaningful vector representations.
3.  **Cosine Similarity**: How we measure the similarity between those vectors.

**Prerequisites:**
- You need an OpenAI API Key saved in a `.env` file in the parent directory.
- You need to install the following libraries:
```bash
pip install openai tiktoken numpy python-dotenv
```

In [ ]:
import os
import numpy as np
from openai import OpenAI
import tiktoken
import openai


# Option 1
# Load from Google Secrets Manager

# from google.colab import userdata

# try:
#     api_key = userdata.get('OPENAI_API_KEY')
# except Exception as e:
#     print("Error retrieving API key. Make sure you set 'OPENAI_API_KEY' in Colab Secrets.")
#     api_key = None

# Option 2
# Load environment variables
# from dotenv import load_dotenv
# load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [ ]:
from google.colab import userdata
userdata.get('OPENAI_API_KEY')

## 1. Tokenization

Text isn't stored as "words" in a model; it's broken down into **tokens**. A token can be a word, part of a word, or even a punctuation mark.

We will use `tiktoken`, the library used by OpenAI models. We'll use the encoding `cl100k_base` which is used by GPT-4 and GPT-3.5.

In [2]:
def show_tokens(text):
    encoding = tiktoken.get_encoding("cl100k_base")
    tokens = encoding.encode(text)
    
    print(f"Original Text: \"{text}\"")
    print(f"Token IDs: {tokens}")
    print(f"Count: {len(tokens)}")
    print("-" * 30)
    
    # Decode back to verify
    print("Token Breakdowns:")
    for t in tokens:
        decoded = encoding.decode([t])
        print(f"ID: {t:<8} -> '{decoded}'")
        
show_tokens("Learning Generative AI is fun!")
print("\n" + "="*40 + "\n")
show_tokens("I love programming.")

Original Text: "Learning Generative AI is fun!"
Token IDs: [48567, 2672, 1413, 15592, 374, 2523, 0]
Count: 7
------------------------------
Token Breakdowns:
ID: 48567    -> 'Learning'
ID: 2672     -> ' Gener'
ID: 1413     -> 'ative'
ID: 15592    -> ' AI'
ID: 374      -> ' is'
ID: 2523     -> ' fun'
ID: 0        -> '!'


Original Text: "I love programming."
Token IDs: [40, 3021, 15840, 13]
Count: 4
------------------------------
Token Breakdowns:
ID: 40       -> 'I'
ID: 3021     -> ' love'
ID: 15840    -> ' programming'
ID: 13       -> '.'


## 2. Text Embeddings

An **embedding** is a list of floating-point numbers (a vector) that represents the *semantic meaning* of the text. 
- Texts with similar meanings will have similar embedding vectors.
- We will use OpenAI's `text-embedding-3-small` model.

In [2]:
def get_embedding(text, model="text-embedding-3-small"):
    text = text.replace("\n", " ")
    return client.embeddings.create(input=[text], model=model).data[0].embedding

# Example
word = "Apple"
embedding = get_embedding(word)

print(f"Embedding for '{word}':")
print(f"Dimensions (Vector Length): {len(embedding)}")
print(f"First 5 values: {embedding[:5]}...")

Embedding for 'Apple':
Dimensions (Vector Length): 1536
First 5 values: [0.009192094206809998, -0.035097088664770126, -0.024993380531668663, 0.03975643962621689, 0.001820059958845377]...


## 3. Cosine Similarity

To measure how similar two vectors (A and B) are, we use **Cosine Similarity**.
It measures the cosine of the angle between two vectors.

$$ \text{similarity} = \cos(\theta) = \frac{A \cdot B}{\|A\| \|B\|} $$

- **1.0**: The vectors are identical (same direction).
- **0.0**: The vectors are orthogonal (unrelated).
- **-1.0**: The vectors are opposite.

In [4]:
def cosine_similarity(a, b):
    a = np.array(a)
    b = np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def compare_texts(text1, text2):
    emb1 = get_embedding(text1)
    emb2 = get_embedding(text2)
    similarity = cosine_similarity(emb1, emb2)
    print(f"'{text1}' vs '{text2}' \nSimilarity: {similarity:.4f}\n")

### Demo: Comparing Concepts

In [5]:
# Related words
compare_texts("Apple", "Banana")
compare_texts("Apple", "iPhone")

# Unrelated words
compare_texts("Apple", "Truck")
compare_texts("Banana", "SpaceX")

'Apple' vs 'Banana' 
Similarity: 0.3862

'Apple' vs 'iPhone' 
Similarity: 0.6966

'Apple' vs 'Truck' 
Similarity: 0.2606

'Banana' vs 'SpaceX' 
Similarity: 0.1750



### Demo: Comparing Sentences
Notice how semantic meaning is captured even if words are different.

In [6]:
s1 = "The cat is happy."
s2 = "The kitten is joyful."  # Semantically same, different words
s3 = "The stock market crashed." # Completely different

compare_texts(s1, s2)
compare_texts(s1, s3)

'The cat is happy.' vs 'The kitten is joyful.' 
Similarity: 0.7956

'The cat is happy.' vs 'The stock market crashed.' 
Similarity: 0.0195



## Student Activity 1: Tokenizer Playground

Try changing the text below to include:
1.  Emojis (e.g., 🍎, 🚀)
2.  Different languages (e.g., Spanish, Arabic, Japanese)
3.  Made-up words

Observe how many tokens are used for each compared to the number of characters.

In [7]:
# EXPERIMENT HERE
my_text = "Hello world! 🌍"

show_tokens(my_text)

Original Text: "Hello world! 🌍"
Token IDs: [9906, 1917, 0, 11410, 234, 235]
Count: 6
------------------------------
Token Breakdowns:
ID: 9906     -> 'Hello'
ID: 1917     -> ' world'
ID: 0        -> '!'
ID: 11410    -> ' �'
ID: 234      -> '�'
ID: 235      -> '�'


## Student Activity 2: Mini Semantic Search Engine

We have a list of sentences (our "database").
**Your Task:**
1.  Run the code to see which sentences are most similar to the query.
2.  **Challenge**: Add your own sentences to the `corpus` list and try new queries to see if the search works as expected!

In [8]:
# 1. Our "Database" of documents
corpus = [
    "The quick brown fox jumps over the lazy dog.",
    "I love eating pizza on Fridays.",
    "Space exploration drives scientific innovation.",
    "Artificial Intelligence is changing the world.",
    "Soccer is the most popular sport globally.",
    "The weather today is sunny and warm.",
    "Cats are independent and curious animals."
]

# 2. Your Query (Try changing this!)
query = "tell me about food"

# 3. Search Function
def search(query, documents):
    print(f"Query: '{query}'\n")
    query_embedding = get_embedding(query)
    
    results = []
    for doc in documents:
        doc_embedding = get_embedding(doc)
        similarity = cosine_similarity(query_embedding, doc_embedding)
        results.append((doc, similarity))
    
    # Sort by similarity (highest first)
    results.sort(key=lambda x: x[1], reverse=True)
    
    # Print top 3 matches
    print("Top 3 Matches:")
    for doc, score in results[:3]:
        print(f"[{score:.4f}] {doc}")

# Run the search
search(query, corpus)

Query: 'tell me about food'

Top 3 Matches:
[0.3003] I love eating pizza on Fridays.
[0.1161] Cats are independent and curious animals.
[0.1123] The quick brown fox jumps over the lazy dog.
